### Sparse heat-equation systems on the periodic unit box

- The stencil names are their 2D names. In $d$ dimensions, the stencil sizes are $2d+1$, $4d+1$, and $3^d$, respectively.

- The box extension is
  $$
  L
  =
  \frac{6}{h^2}
  \left[
  \left(I+\frac{T}{6}\right)
  \otimes \cdots \otimes
  \left(I+\frac{T}{6}\right)
  - I
  \right],
  $$
  where $T$ is the dimensionless 1D periodic second-difference operator.

- The time-discretization matrices are
  $$
  A = I-\theta\,\Delta t\,L,
  \qquad
  B = I+(1-\theta)\,\Delta t\,L.
  $$

- Each time step solves
  $$
  A u^{k+1} = B u^k.
  $$

- Use $\theta=0.5$ for Crank--Nicolson or $\theta=1$ for backward Euler.

#### Notes

- For $0.5 \leq \theta \leq 1$, the time update is unconditionally stable in $L^2$.
- Values $0 < \theta < 0.5$ require a stencil-dependent time-step restriction. This script does not check that restriction.
- Forward Euler, corresponding to $\theta=0$, is excluded.

In [ ]:
from functools import reduce
import numpy as np
from scipy.sparse import csr_matrix, diags, eye, kron
from scipy.sparse.linalg import spsolve


def heat_matrices(
    d: int = 2,
    h: float = 1 / 32,
    dt: float = 1e-3,
    stencil: str = "5-point",
    theta: float = 0.5,
) -> tuple[csr_matrix, csr_matrix]:
    """Return (A, B) for A @ u_next = B @ u on periodic [0,1)^d.

    Use d=1, 2, or 3; h=1/n for an integer n>=5; dt>0; 0<theta<=1.
    All stencil names refer to their 2D forms. No forcing is included.
    """
    if not isinstance(d, (int, np.integer)) or d not in (1, 2, 3):
        raise ValueError("d must be 1, 2, or 3.")
    if not np.isfinite(h) or h <= 0 or not np.isfinite(dt) or dt <= 0:
        raise ValueError("h and dt must be positive and finite.")
    if not np.isfinite(theta) or not 0 < theta <= 1:
        raise ValueError("Use 0 < theta <= 1; theta >= 0.5 avoids a stability restriction.")
    n = round(1 / h)
    if n < 5 or not np.isclose(n * h, 1.0, rtol=1e-12, atol=1e-12):
        raise ValueError("Use h=1/n for an integer n>=5.")
    if stencil not in ("5-point", "9-point-cross", "9-point-box"):
        raise ValueError("Unknown stencil.")

    I = eye(n, format="csr")
    IN = eye(n**d, format="csr")
    S = diags([1.0, 1.0], [1, 1 - n], shape=(n, n), format="csr")
    T = S + S.T - 2 * I  # Periodic [1, -2, 1], without 1/h**2.

    def tensor(factors):
        result = factors[0]
        for matrix in factors[1:]:
            result = kron(result, matrix, format="csr")
        return result

    if stencil == "9-point-box":
        Q = I + T / 6
        L = 6 * (tensor([Q] * d) - IN) / h**2
    else:
        if stencil == "9-point-cross":
            T = T - (T @ T) / 12  # [-1, 16, -30, 16, -1] / 12.
        L = csr_matrix((n**d, n**d))
        for axis in range(d):
            factors = [I] * d
            factors[axis] = T
            L += tensor(factors) / h**2

    A = (IN - theta * dt * L).tocsr()
    B = (IN + (1 - theta) * dt * L).tocsr()
    for M in (A, B):
        M.eliminate_zeros()
        M.sort_indices()
    return A, B

theta=0.5; 10 systems; A.shape=(1024, 1024); A.nnz=9216
Final relative residual: 7.911e-16


In [ ]:
A, B = heat_matrices(
    d=2,
    h=1 / 32,
    dt=1e-3,
    stencil="9-point-box",
    theta=0.5,              # 0.5: Crank-Nicolson; 1.0: backward Euler
)

u = np.random.default_rng(0).standard_normal(A.shape[0])
u -= u.mean()

def f(t):
    # One forcing value per grid point; here f(x, t) = 1 everywhere.
    return np.ones(A.shape[0])

systems = []
for k in range(10):
    b = B @ u
    systems.append((A, b))
    u = spsolve(A, b)

In [ ]:
from itertools import product
from pathlib import Path
from time import perf_counter
import sys
from scipy.sparse.linalg import cg

# Find baselines.py from the notebook, project, or workspace directory.
project_dirs = [Path.cwd(), *Path.cwd().parents,
                Path.cwd() / "project" / "preconditioner-survey"]
project_root = next(path for path in project_dirs if (path / "baselines.py").is_file())
sys.path.insert(0, str(project_root))
import baselines

# This full sweep is large; shorten these lists for a quick run.
ds = [1, 2, 3]
hs = [(1 / 2)**p for p in range(4, 10)]
dts = [(1e-1)**t for t in range(1, 5)]
stencils = ["5-point", "9-point-cross", "9-point-box"]
thetas = [0.5 + 0.1 * i for i in range(6)]

preconditioners = ["plain", "jacobi", "block_jacobi", "richardson", "IC0"]
rtol, maxiter = 1e-8, 1000

# Each constructor returns an approximate inverse for CG's M argument.
constructors = {
    "plain": baselines.identity,
    "jacobi": baselines.jacobi,
    "block_jacobi": baselines.block_jacobi,
    # Damping keeps Richardson positive definite for these heat stencils.
    "richardson": lambda A: baselines.richardson_jacobi(A, omega=0.5),
    "IC0": baselines.incomplete_cholesky,  # Requires the optional ilupp package.
}

systems = []
results = []
for d, h, dt, stencil, theta in product(ds, hs, dts, stencils, thetas):
    A, B = heat_matrices(d, h, dt, stencil, theta)

    # Build each preconditioner once and reuse it for all time steps.
    M, setup_seconds = {}, {}
    for name in preconditioners:
        start = perf_counter()
        M[name] = constructors[name](A)
        setup_seconds[name] = perf_counter() - start

    # Reuse the same initial condition when comparing settings on the same grid.
    u0 = np.random.default_rng(0).standard_normal(A.shape[0])
    u0 -= u0.mean()
    u = u0.copy()
    f = np.zeros(A.shape[0])  # Time-independent heat source; change to add heat.

    for k in range(20):
        b = B @ u + dt * f  # For constant f, the theta weights sum to one.
        systems.append((A, b))  # A is shared; each step has a new right-hand side.
        b_norm = np.linalg.norm(b)
        u_next = None

        # Every baseline solves the same system from CG's default zero guess.
        for name in preconditioners:
            iterations = []  # The callback runs once per CG iteration.
            start = perf_counter()
            x, info = cg(A, b, M=M[name], rtol=rtol, atol=0.0, maxiter=maxiter,
                         callback=lambda _: iterations.append(None))
            solve_seconds = perf_counter() - start
            residual = np.linalg.norm(A @ x - b)
            results.append({
                "d": d, "h": h, "dt": dt, "stencil": stencil, "theta": theta,
                "step": k + 1, "preconditioner": name,
                "iterations": len(iterations), "info": info, "converged": info == 0,
                "relative_residual": residual / b_norm if b_norm else residual,
                "setup_seconds": setup_seconds[name],  # Paid once per matrix/baseline.
                "solve_seconds": solve_seconds,
            })
            if info == 0 and u_next is None:
                u_next = x

        if u_next is None:
            raise RuntimeError(f"No baseline converged at step {k + 1}.")
        u = u_next  # Advance with the first converged solution.

print(f"Generated {len(systems)} systems; recorded {len(results)} CG solves.")